In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LOADING AND MERGING DATASETS
print("Loading datasets for evaluation...")

# Loading the Ground Truth (API) data
# Replacing with the exact path/name of API extracted file
api_df = pd.read_excel('families_mapped_to_images_filtered_for_prompt.xlsx')

# Loading the Local Model Predictions
model_df = pd.read_excel('families_mapped_to_images_with_features.xlsx')

# Merging the two dataframes on the unique identifiers. 
# Suffixes will automatically append '_api' and '_model' to overlapping columns
merged_df = pd.merge(
    api_df, 
    model_df, 
    on=['Property_ID', 'hasfamilyid', 'Image_Name'], 
    how='inner',
    suffixes=('_api', '_model')
)

print(f"Successfully matched {len(merged_df)} images for comparison.\n")

# CATEGORICAL ACCURACY EVALUATION
CATEGORICAL_COLS = [
    'Image_Context', 'Housing_Category', 'Ext_Stories', 
    'Ext_Roof_Type', 'Ext_Wall_Type', 'Ext_Structural_Condition', 
    'Int_Floor_Material', 'Int_Wall_Finish'
]

print("       CATEGORICAL ACCURACY (Exact Match)         ")
for col in CATEGORICAL_COLS:
    # Convert to string to prevent mismatch errors between NaNs and 'N/A' strings
    col_api = merged_df[f"{col}_api"].fillna('N/A').astype(str)
    col_model = merged_df[f"{col}_model"].fillna('N/A').astype(str)
    
    matches = (col_api == col_model).sum()
    accuracy = (matches / len(merged_df)) * 100
    print(f"{col.ljust(35)}: {accuracy:.2f}%")

# INDIVIDUAL ASSET EVALUATION

print("        VISIBLE ASSETS EVALUATION (Per Asset)       ")

ASSET_LIST = ['AC', 'Tractor', 'Truck', 'Two-Wheeler', 'Four-Wheeler']

def check_asset_presence(asset_string, target_asset):
    """Safely checks if a specific asset is inside the comma-separated string."""
    if pd.isna(asset_string) or asset_string == '[]':
        return False
    return target_asset in str(asset_string)

for asset in ASSET_LIST:
    # Creating True/False arrays indicating if the asset was found
    api_has_asset = merged_df['Visible_Assets_For_Income_api'].apply(lambda x: check_asset_presence(x, asset))
    model_has_asset = merged_df['Visible_Assets_For_Income_model'].apply(lambda x: check_asset_presence(x, asset))
    
    # Calculating how often the Model agreed with the API (both True or both False)
    asset_matches = (api_has_asset == model_has_asset).sum()
    asset_accuracy = (asset_matches / len(merged_df)) * 100
    
    # Optional: Counting how many times the API actually saw this asset to give context
    api_count = api_has_asset.sum()
    
    print(f"{asset.ljust(25)} (n={str(api_count).ljust(3)}): {asset_accuracy:.2f}% Match")


# STRUCTURAL SCORE EVALUATION
print("             STRUCTURAL SCORE EVALUATION            ")

# Ensuring numeric types
api_scores = pd.to_numeric(merged_df['Overall_Structural_Score_api'], errors='coerce').fillna(0)
model_scores = pd.to_numeric(merged_df['Overall_Structural_Score_model'], errors='coerce').fillna(0)

# Mean Absolute Error (MAE)
mae = np.abs(api_scores - model_scores).mean()
print(f"{'Mean Absolute Error (MAE)'.ljust(35)}: {mae:.4f}")

# Threshold Accuracy
score_diff = np.abs(api_scores - model_scores)
within_threshold = (score_diff <= 0.10).sum()
threshold_accuracy = (within_threshold / len(merged_df)) * 100
print(f"{'Score Accuracy (Within ±0.1)'.ljust(35)}: {threshold_accuracy:.2f}%")

print("\nEvaluation Complete.")

Loading datasets for evaluation...
Successfully matched 2301 images for comparison.

       CATEGORICAL ACCURACY (Exact Match)         
Image_Context                      : 48.11%
Housing_Category                   : 81.66%
Ext_Stories                        : 87.96%
Ext_Roof_Type                      : 95.48%
Ext_Wall_Type                      : 97.35%
Ext_Structural_Condition           : 78.66%
Int_Floor_Material                 : 95.87%
Int_Wall_Finish                    : 97.65%
        VISIBLE ASSETS EVALUATION (Per Asset)       
AC                        (n=814): 83.88% Match
Tractor                   (n=16 ): 83.31% Match
Truck                     (n=6  ): 99.26% Match
Two-Wheeler               (n=382): 83.40% Match
Four-Wheeler              (n=395): 79.18% Match
             STRUCTURAL SCORE EVALUATION            
Mean Absolute Error (MAE)          : 0.0869
Score Accuracy (Within ±0.1)       : 66.93%

Evaluation Complete.


In [2]:
import pandas as pd

# FILE CONFIGURATION
MASTER_DATA_FILE = "processed_district_data_with_property_ids.xlsx"
FEATURES_DATA_FILE = "families_mapped_to_images_filtered_for_prompt.xlsx"
OUTPUT_FILE = "master_dataset_with_ai_features.xlsx"

def integrate_datasets():
    print("--- 🔄 INITIATING DATA INTEGRATION ---")
    
    # LOADING DATASETS
    try:
        print(f"Loading master dataset: {MASTER_DATA_FILE}...")
        df_master = pd.read_excel(MASTER_DATA_FILE)
        
        print(f"Loading AI features dataset: {FEATURES_DATA_FILE}...")
        df_features = pd.read_excel(FEATURES_DATA_FILE)
    except FileNotFoundError as e:
        print(f"❌ Error loading files: {e}")
        return

    # MERGING THE DATA
    print("Merging datasets on 'Property_ID' and 'hasfamilyid'...")
    
    # A 'left' merge ensures we don't drop any families from the master dataset
    df_merged = pd.merge(
        df_master, 
        df_features, 
        on=['Property_ID', 'hasfamilyid'], 
        how='left'
    )
    
    # Optional: Filling missing AI features with a default string to avoid downstream errors
    ai_columns = [col for col in df_features.columns if col not in ['Property_ID', 'hasfamilyid']]
    df_merged[ai_columns] = df_merged[ai_columns].fillna('Null')

    # EXPORTING FINAL DATASET
    print(f"Saving the integrated dataset to {OUTPUT_FILE}...")
    df_merged.to_excel(OUTPUT_FILE, index=False)
    
    print("\n✅ Integration complete!")
    print(f"Total records in master dataset: {len(df_master)}")
    print(f"Total records in merged dataset: {len(df_merged)}")

if __name__ == "__main__":
    integrate_datasets()

--- 🔄 INITIATING DATA INTEGRATION ---
Loading master dataset: processed_district_data_with_property_ids.xlsx...
Loading AI features dataset: families_mapped_to_images_filtered_for_prompt.xlsx...
Merging datasets on 'Property_ID' and 'hasfamilyid'...
Saving the integrated dataset to master_dataset_with_ai_features.xlsx...

✅ Integration complete!
Total records in master dataset: 2301
Total records in merged dataset: 2301
